# 02. Pemodelan ARIMA

Membangun model ARIMA mengikuti alur skripsi: Box-Cox, differencing 2, estimasi lima kandidat, uji signifikansi parameter, uji Ljung-Box, uji normalitas residual, lalu pemilihan model dengan RMSE/MAPE training terkecil.

In [1]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src") if Path.cwd().name == "notebooks" else str(Path.cwd() / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
from kurs_forecast.data import load_kurs, split_train_test
serie = load_kurs(); train, test = split_train_test(serie)
print(f"Total {len(serie)} minggu: training {len(train)}, testing {len(test)}")

Total 105 minggu: training 84, testing 21


In [2]:
from kurs_forecast.models import pilih_arima_terbaik, forecast_arima
terbaik, tabel = pilih_arima_terbaik(train)
tabel.round(4)

,model,parameter,semua_signifikan,white_noise,normal_ks,AIC,RMSE_train,MAPE_train,terpilih
0,"ARIMA(0, 2, 1)",ma.L1=-0.9835,True,True,True,23.1028,129.3987,0.6456,True
1,"ARIMA(1, 2, 0)",ar.L1=-0.6057,True,True,True,44.3817,147.2315,0.7626,False
2,"ARIMA(2, 2, 0)","ar.L1=-0.8058, ar.L2=-0.3203",True,True,True,37.8307,139.1291,0.7224,False
3,"ARIMA(1, 2, 1)","ar.L1=-0.0983, ma.L1=-0.9639",False,True,True,24.4271,129.0110,0.6423,False
4,"ARIMA(2, 2, 1)","ar.L1=-0.0656, ar.L2=0.1552, ma.L1=-0.9982",False,True,True,24.6332,127.3700,0.6387,False


Tabel di atas mereplikasi Tabel 4.1 skripsi: hanya ARIMA (0,2,1), (1,2,0), dan (2,2,0) yang seluruh parameternya signifikan. ARIMA (0,2,1) terpilih karena RMSE training terkecil (skripsi: 125,78; di sini sedikit berbeda karena estimator statsmodels dan R tidak identik).

In [3]:
print('Model terpilih: ARIMA', terbaik.order)
print('Parameter:', terbaik.params, 'p-value:', terbaik.pvalues)
terbaik.ljung_box.round(4)

Model terpilih: ARIMA (0, 2, 1)
Parameter: {'ma.L1': np.float64(-0.9835102146007577)} p-value: {'ma.L1': np.float64(2.706487063984585e-45)}


,lb_stat,lb_pvalue,white_noise
lag,,,
12,9.1914,0.6865,True
24,17.8395,0.8107,True
36,33.1631,0.6042,True
48,39.7101,0.7971,True


In [4]:
terbaik.normalitas

{'ks_statistic': 0.08578507862651974,
 'ks_pvalue': 0.5531969070772292,
 'shapiro_statistic': 0.9683540031186011,
 'shapiro_pvalue': 0.04035702613411827,
 'normal_ks': True,
 'normal_shapiro': False}

In [5]:
from kurs_forecast.plots import plot_residual, plot_fit_train
d = terbaik.order[1]
plot_residual(terbaik.model.resid[d:], f'Diagnostik residual ARIMA{terbaik.order}');
plot_fit_train(train, terbaik.fitted_train, f'ARIMA{terbaik.order}', offset=d);

## Prediksi data testing
Peramalan 21 minggu ke depan lalu dikembalikan ke skala rupiah. Seperti temuan skripsi, ARIMA menghasilkan garis yang hampir datar karena model MA(1) dengan differencing 2 hanya memproyeksikan tren lokal terakhir.

In [6]:
from kurs_forecast.metrics import all_metrics
pred = forecast_arima(terbaik, len(test))
print(all_metrics(test.values, pred))
plt.figure(figsize=(10,4)); plt.plot(test.index, test.values, 'k-o', ms=4, label='Aktual'); plt.plot(test.index, pred, 'g--s', ms=3, label=f'ARIMA{terbaik.order}'); plt.legend(); plt.grid(alpha=.3); plt.title('Prediksi testing ARIMA');

{'RMSE': 421.04860720742664, 'MAE': 377.0157364940503, 'MAPE': 2.4733691207058457}
